# GAT Fraud Detection — IEEE-CIS (GTAN Methodology)

**Architecture basis:** `architecture_justification_v2.md` §4.4 + §4.2  
**Data pipeline basis:** `experiment_tcn.ipynb` (same preprocessing, no sequence reshaping)  
**Framework:** PyTorch + DGL  

## What This Notebook Does

- Replicates the exact 263-feature preprocessing pipeline from `experiment_tcn`
- Builds a **GTAN-style homogeneous transaction graph** (temporal/relational edges by shared attributes)
- Trains a **2-layer GAT** (4 heads → 256-d → 64-d) as the relational view of the fraud pipeline
- Uses **DGL mini-batch training** with `NeighborSampler([10, 5])` for scalability
- Outputs 64-d node embeddings, compatible with the downstream Gated Fusion + CaT-GNN pipeline

## Why Not the TCN-Pipeline GAT

The GAT section in `tcn-training-pipeline.ipynb` has three fundamental problems that make it unusable:

1. **Single-node graphs per sample** — the `FusionDataGenerator` feeds each sample as its own standalone single-node graph. GAT on a single node is mathematically equivalent to an MLP (no neighbors = no attention). The k-NN adjacency is built but never used for actual message passing.
2. **k-NN graph misses fraud-ring structure** — Euclidean distance on features connects *similar* transactions, not *related* ones. GTAN's insight is to connect transactions sharing `uid/card1/P_emaildomain/DeviceInfo` to capture fraud rings (same stolen card, same compromised device).
3. **TF/Keras framework** — incompatible with the DGL/PyTorch-based CaT-GNN and Gated Fusion architecture. Output is also 32-d, mismatching the required 64-d fusion inputs.

## Architecture Summary

```
590K transaction nodes, each with 263-d feature vector
  │
  ├─ GTAN Graph: edges via uid / card1 / P_emaildomain / DeviceInfo (K=3 temporal)
  │
  ├─ GAT Layer 1: GATConv(263 → 64, heads=4) → flatten → (N, 256)
  │   BatchNorm1d(256) + ELU + Dropout(0.3)
  │
  ├─ GAT Layer 2: GATConv(256 → 64, heads=1) → squeeze → (N, 64)
  │   BatchNorm1d(64)
  │
  └─ FraudClassifier: Linear(64→32) → ReLU → Dropout(0.2) → Linear(32→1)
```

In [ ]:
# Install DGL — uncomment the correct line for your environment
# Kaggle (PyTorch 2.1, CUDA 12.1)
# !pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html -q
# CPU-only / local
# !pip install dgl -q
!pip install dgl -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn import GATConv
import dgl.dataloading

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')
import os, json, gc, time
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'DGL version: {dgl.__version__}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Graph construction (GTAN methodology — arch_justification_v2 §4.2)
GRAPH_K_NEIGHBORS = 3        # K=3 past temporal neighbors per relation (matches GTAN edge_per_trans=3)
GRAPH_RELATION_COLS = ['uid', 'card1', 'P_emaildomain', 'DeviceInfo']
MAX_GROUP_SIZE = 5000        # Skip hub groups (e.g. gmail.com) to avoid degree imbalance

# GAT model (arch_justification_v2 §4.4)
HIDDEN_DIM = 64              # Per-head output dim → L1=64×4=256, L2=64×1=64
NUM_HEADS_L1 = 4             # 4 heads: matches CaT-GNN paper; required for downstream attention scoring
FEAT_DROP = 0.3
ATTN_DROP = 0.3

# Training hyperparameters — ALL from experiment_tcn.ipynb
BATCH_SIZE   = 512           # experiment_tcn: BATCH_SIZE = 512
LR           = 0.0005        # experiment_tcn: LEARNING_RATE = 0.0005
EPOCHS       = 200           # experiment_tcn: EPOCHS = 200
PATIENCE_ES  = 50            # experiment_tcn: EarlyStopping patience = 50 (monitor val_auc)
PATIENCE_LR  = 10            # experiment_tcn: ReduceLROnPlateau patience = 10
LR_FACTOR    = 0.5           # experiment_tcn: factor = 0.5
MIN_LR       = 1e-7          # experiment_tcn: min_lr = 1e-7
FAN_OUT      = [10, 5]       # Neighbor sampling fan-out per layer

# Paths
SAVE_DIR = 'gat_results'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'models'), exist_ok=True)

print('Configuration:')
print(f'  [ARCH v2]  Graph K={GRAPH_K_NEIGHBORS}, Relations: {GRAPH_RELATION_COLS}')
print(f'  [ARCH v2]  GAT: L1={HIDDEN_DIM}×{NUM_HEADS_L1}h={HIDDEN_DIM*NUM_HEADS_L1}-d, L2={HIDDEN_DIM}×1h={HIDDEN_DIM}-d')
print(f'  [TCN REF]  lr={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}')
print(f'  [TCN REF]  EarlyStopping patience={PATIENCE_ES}, ReduceLROnPlateau patience={PATIENCE_LR}')

---
## 1. Data Loading & Preprocessing

Identical to `experiment_tcn.ipynb` — same column selection, encoding functions, UID construction, and aggregation features. The only difference: we **do not reshape to sequences**. GAT uses the flat 263-feature vector as node features.

We also retain the `uid` string column temporarily for graph construction before dropping it from node features.

In [ ]:
# Load raw data
X_train  = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
X_test   = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_transaction.csv')
train_id = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_identity.csv')
test_id  = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_identity.csv')

# Normalize identity column names: Kaggle ships id-01 to id-38 (dash) in some
# splits and id_01 to id_38 (underscore) in others. Standardize before merge.
for df in [train_id, test_id, X_train, X_test]:
    df.columns = [c.replace("-", "_") if c.startswith("id") else c
                  for c in df.columns]

# Merge identity on TransactionID column (correct approach for IEEE-CIS)
X_train = X_train.merge(train_id, on="TransactionID", how="left")
X_test  = X_test.merge(test_id,  on="TransactionID", how="left")

# Extract labels then reset both to clean 0-based index
y_train = X_train["isFraud"].copy()
del X_train["isFraud"]

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f"Train: {X_train.shape},  Test: {X_test.shape}")
print(f"Fraud rate: {y_train.mean()*100:.2f}%")
print(f"id_ cols in train: {sum(1 for c in X_train.columns if c.startswith('id_'))}  "
      f"id_ cols in test: {sum(1 for c in X_test.columns if c.startswith('id_'))}")

In [ ]:
# Column definitions (same as experiment_tcn) ─────────────────────────────────
# V-columns selected by NaN-correlation clustering (experiment_tcn exact list)
V_COLS  = [1,3,4,6,8,11]
V_COLS += [13,14,17,20,23,26,27,30]
V_COLS += [36,37,40,41,44,47,48]
V_COLS += [54,56,59,62,65,67,68,70]
V_COLS += [76,78,80,82,86,88,89,91]
V_COLS += [107,108,111,115,117,120,121,123]
V_COLS += [124,127,129,130,136]
V_COLS += [138,139,142,147,156,162]
V_COLS += [165,160,166]
V_COLS += [178,176,173,182]
V_COLS += [187,203,205,207,215]
V_COLS += [169,171,175,180,185,188,198,210,209]
V_COLS += [218,223,224,226,228,229,235]
V_COLS += [220,221,234,238,250,271]          # present in experiment_tcn, was missing here
V_COLS += [240,258,257,253,252,260,261]
V_COLS += [264,266,267,274,277]
V_COLS += [294,284,285,286,291,297]
V_COLS += [303,305,307,309,310,320]
V_COLS += [281,283,289,296,301,314]

# Keep only selected V columns — drop the rest
all_cols = list(X_train.columns)
v_to_drop = [c for c in all_cols if c.startswith('V') and c not in [f'V{i}' for i in V_COLS]]
X_train.drop(columns=v_to_drop, inplace=True)
X_test.drop(columns=v_to_drop, inplace=True)

# Categorical columns (experiment_tcn str_type list — underscored after normalization)
CAT_COLS = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'id_12','id_15','id_16','id_23','id_27','id_28','id_29',
    'id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38',
    'DeviceType','DeviceInfo'
]

print(f'V_COLS count: {len(set(V_COLS))}')
print(f'Shape after V-col drop: {X_train.shape}')

In [ ]:
# Sort by TransactionDT — critical for GTAN temporal graph construction.
# Attach y_train as column so both sort together; avoids index-alignment issues.
X_train['__label__'] = y_train.values
X_train = X_train.sort_values('TransactionDT').reset_index(drop=True)
y_train = X_train.pop('__label__').astype('int8')
X_test  = X_test.sort_values('TransactionDT').reset_index(drop=True)

# day column (used for UID and D-transform)
X_train['day'] = X_train['TransactionDT'] / np.float32(24 * 60 * 60)
X_test['day']  = X_test['TransactionDT']  / np.float32(24 * 60 * 60)

# D-column transform — experiment_tcn exact approach:
#   formula : D_new = D_old - day   (same sign as experiment_tcn)
#   SKIP D1, D2, D3, D5, D9 — these keep their original values.
#   D1 in particular must NOT be transformed: UID uses floor(day - D1_orig).
#   If D1 were transformed to (day - D1_orig) first, UID would become floor(D1_orig) — wrong grouping.
D_SKIP = {1, 2, 3, 5, 9}
for i in range(1, 16):
    col = f'D{i}'
    if i in D_SKIP:
        continue          # keep original value
    if col in X_train.columns:
        X_train[col] = X_train[col] - X_train['day']
    if col in X_test.columns:
        X_test[col]  = X_test[col]  - X_test['day']

# DT_M — experiment_tcn exact formula (calendar month, not 30-day approximation)
import datetime
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
for df in [X_train, X_test]:
    dt_series = df['TransactionDT'].apply(
        lambda x: START_DATE + datetime.timedelta(seconds=x))
    df['DT_M'] = (dt_series.dt.year - 2017) * 12 + dt_series.dt.month

# Factorize categoricals + shift numerics positive, NaN -> -1
SKIP = {'TransactionAmt', 'TransactionDT', 'day', 'DT_M'}
shared_cols = set(X_train.columns) & set(X_test.columns)

for f in list(X_train.columns):
    if (str(X_train[f].dtype) == 'category') or (X_train[f].dtype == 'object'):
        if f in shared_cols:
            df_comb = pd.concat([X_train[f], X_test[f]], axis=0)
            df_comb, _ = df_comb.factorize(sort=True)
            if df_comb.max() > 32000:
                X_train[f] = df_comb[:len(X_train)].astype('int32')
                X_test[f]  = df_comb[len(X_train):].astype('int32')
            else:
                X_train[f] = df_comb[:len(X_train)].astype('int16')
                X_test[f]  = df_comb[len(X_train):].astype('int16')
        else:
            X_train[f], _ = X_train[f].factorize(sort=True)
            X_train[f] = X_train[f].astype('int16')
    elif f not in SKIP:
        if f in shared_cols:
            mn = np.min((X_train[f].min(), X_test[f].min()))
            X_train[f] -= np.float32(mn)
            X_test[f]  -= np.float32(mn)
            X_train[f].fillna(-1, inplace=True)
            X_test[f].fillna(-1, inplace=True)
        else:
            mn = X_train[f].min()
            X_train[f] -= np.float32(mn)
            X_train[f].fillna(-1, inplace=True)

print(f'Sorted + preprocessed. X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Fraud rate after sort: {y_train.mean()*100:.2f}%  (should be unchanged)')
print(f'DT_M range: {X_train["DT_M"].min()} to {X_train["DT_M"].max()}  (should be ~12 to 24)')

In [ ]:
# ── Encoding helper functions (same as experiment_tcn) ─────────────────────────
def encode_LE(col, verbose=True, df1=None, df2=None):
    if df1 is None: df1 = X_train
    if df2 is None: df2 = X_test
    df_comb = pd.concat([df1[col], df2[col]], axis=0)
    df_comb, _ = df_comb.factorize(sort=True)
    if df_comb.max() > 32000:
        df1[col] = df_comb[:len(df1)].astype('int32')
        df2[col] = df_comb[len(df1):].astype('int32')
    else:
        df1[col] = df_comb[:len(df1)].astype('int16')
        df2[col] = df_comb[len(df1):].astype('int16')
    if verbose: print(col, ', ', end='')

def encode_FE(df1, df2, cols):
    for col in cols:
        df = pd.concat([df1[col], df2[col]])
        vc = df.value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1
        nm = col + '_FE'
        df1[nm] = df1[col].map(vc).astype('float32')
        df2[nm] = df2[col].map(vc).astype('float32')
        print(nm, ', ', end='')

def encode_AG(main_columns, uids, aggregations=['mean'], train_df=None, test_df=None,
              fillna=True, usena=False):
    if train_df is None: train_df = X_train
    if test_df  is None: test_df  = X_test
    for main_column in main_columns:
        for col in uids:
            for agg_type in aggregations:
                new_col_name = main_column + '_' + col + '_' + agg_type
                temp_df = pd.concat([train_df[[col, main_column]], test_df[[col, main_column]]])
                if usena:
                    temp_df.loc[temp_df[main_column] == -1, main_column] = np.nan
                temp_df = temp_df.groupby([col])[main_column].agg([agg_type]).reset_index()
                temp_df = temp_df.rename(columns={agg_type: new_col_name})
                temp_df.index = list(temp_df[col])
                temp_df = temp_df[new_col_name].to_dict()
                train_df[new_col_name] = train_df[col].map(temp_df).astype('float32')
                test_df[new_col_name]  = test_df[col].map(temp_df).astype('float32')
                if fillna:
                    train_df[new_col_name].fillna(-1, inplace=True)
                    test_df[new_col_name].fillna(-1, inplace=True)
                print("'" + new_col_name + "'", ', ', end='')

def encode_AG2(main_columns, uids, train_df=None, test_df=None):
    if train_df is None: train_df = X_train
    if test_df  is None: test_df  = X_test
    for main_column in main_columns:
        for col in uids:
            comb = pd.concat([train_df[[col, main_column]], test_df[[col, main_column]]], axis=0)
            mp = comb.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            train_df[col + '_' + main_column + '_ct'] = train_df[col].map(mp).astype('float32')
            test_df[col + '_' + main_column + '_ct']  = test_df[col].map(mp).astype('float32')
            print(col + '_' + main_column + '_ct, ', end='')

def encode_CB(col1, col2, df1=None, df2=None):
    if df1 is None: df1 = X_train
    if df2 is None: df2 = X_test
    nm = col1 + '_' + col2
    df1[nm] = df1[col1].astype(str) + '_' + df1[col2].astype(str)
    df2[nm] = df2[col1].astype(str) + '_' + df2[col2].astype(str)
    encode_LE(nm, verbose=False)

print('Encoding functions defined.')

In [ ]:
# ── Feature engineering (same as experiment_tcn) ──────────────────────────────
# Cents feature
X_train['cents'] = (X_train['TransactionAmt'] - np.floor(X_train['TransactionAmt'])).astype('float32')
X_test['cents']  = (X_test['TransactionAmt']  - np.floor(X_test['TransactionAmt'])).astype('float32')

# Combined columns
encode_CB('card1', 'addr1')
encode_CB('card1_addr1', 'P_emaildomain')

# UID construction: card1_addr1 + floor(day - D1)
X_train['uid'] = (X_train['card1_addr1'].astype(str) + '_' +
                  np.floor(X_train['day'] - X_train['D1']).astype(str))
X_test['uid']  = (X_test['card1_addr1'].astype(str) + '_' +
                  np.floor(X_test['day']  - X_test['D1']).astype(str))

# Save raw uid & relation cols for graph construction BEFORE they get modified
uid_train    = X_train['uid'].copy()
card1_train  = X_train['card1'].copy()
# P_emaildomain and DeviceInfo are already label-encoded integers — fine for groupby
pemail_train = X_train['P_emaildomain'].copy()
device_train = X_train['DeviceInfo'].copy()

print('UID and relation columns saved for graph construction.')
print(f'Unique UIDs: {uid_train.nunique():,}')

In [ ]:
# ── Frequency, aggregation, count-unique encodings (same as experiment_tcn) ───
print('Frequency encoding:')
encode_FE(X_train, X_test, ['addr1', 'card1', 'card2', 'card3', 'P_emaildomain'])
encode_FE(X_train, X_test, ['card1_addr1', 'card1_addr1_P_emaildomain'])
encode_FE(X_train, X_test, ['uid'])

print('\nAggregation features:')
encode_AG(
    ['TransactionAmt', 'D9', 'D11'],
    ['card1', 'card1_addr1', 'card1_addr1_P_emaildomain'],
    ['mean', 'std'], usena=True
)
encode_AG(
    ['TransactionAmt', 'D4', 'D9', 'D10', 'D15'],
    ['uid'], ['mean', 'std'], fillna=True, usena=True
)
encode_AG(
    ['C' + str(x) for x in range(1, 15) if x != 3],
    ['uid'], ['mean'], fillna=True, usena=True
)
encode_AG(
    ['M' + str(x) for x in range(1, 10)],
    ['uid'], ['mean'], fillna=True, usena=True
)
encode_AG(['C14'], ['uid'], ['std'], fillna=True, usena=True)

print('\nCount-unique features:')
encode_AG2(['P_emaildomain', 'dist1', 'DT_M', 'id_02', 'cents'], ['uid'])
encode_AG2(['C13', 'V314'], ['uid'])
encode_AG2(['V127', 'V136', 'V309', 'V307', 'V320'], ['uid'])

# Outsider feature
X_train['outsider15'] = (np.abs(X_train['D1'] - X_train['D15']) > 3).astype('int8')
X_test['outsider15']  = (np.abs(X_test['D1']  - X_test['D15']) > 3).astype('int8')

print('\nDone.')

In [ ]:
# ── Final column selection (same drop list as experiment_tcn) ─────────────────
cols = list(X_train.columns)
DROP_COLS = (
    ['TransactionDT', 'TransactionID'] +
    ['D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14'] +
    ['DT_M', 'day', 'uid'] +
    ['C3', 'M5', 'id_08', 'id_33'] +
    ['card4', 'id_07', 'id_14', 'id_21', 'id_30', 'id_32', 'id_34'] +
    ['id_' + str(x) for x in range(22, 28)]
)
cols = [c for c in cols if c in X_train.columns and c not in DROP_COLS]

X_gat = X_train[cols].copy().fillna(-1).astype('float32').values  # (590540, n_features)
y_gat = y_train.values.astype('int64')

IN_DIM = X_gat.shape[1]
print(f'Node feature matrix: {X_gat.shape}  →  IN_DIM = {IN_DIM}')
print(f'Labels: {y_gat.shape}, fraud rate = {y_gat.mean()*100:.2f}%')

In [ ]:
# ── Stratified split on node IDs (transductive: all nodes in one graph) ────────
all_nids = np.arange(len(y_gat))
train_nids, val_nids = train_test_split(
    all_nids, test_size=0.2, stratify=y_gat, random_state=42
)
train_nids = torch.LongTensor(train_nids)
val_nids   = torch.LongTensor(val_nids)

print(f'Train nodes: {len(train_nids):,} | Val nodes: {len(val_nids):,}')
print(f'Train fraud: {y_gat[train_nids].mean()*100:.2f}% | '
      f'Val fraud: {y_gat[val_nids].mean()*100:.2f}%')

# StandardScaler fitted on train nodes only
scaler = StandardScaler()
X_gat[train_nids.numpy()] = scaler.fit_transform(X_gat[train_nids.numpy()])
X_gat[val_nids.numpy()]   = scaler.transform(X_gat[val_nids.numpy()])

node_features = torch.FloatTensor(X_gat)
node_labels   = torch.LongTensor(y_gat)

print(f'Feature tensor: {node_features.shape}, Labels tensor: {node_labels.shape}')

---
## 2. GTAN-Style Graph Construction

**Methodology** (Xiang et al., AAAI 2023 — `AI4Risk/antifraud`):
- Nodes: one per transaction (homogeneous)
- Relation columns: `uid`, `card1`, `P_emaildomain`, `DeviceInfo`
- For each relation, group transactions sharing the same attribute value
- Within each group (sorted by TransactionDT), for each transaction connect to K=3 most recent past transactions → directed temporal edges (past → future)
- Add reverse edges (bidirectional) + self-loops for GNN stability

**Key differences from TCN-pipeline kNN graph:**
| | kNN (TCN pipeline) | GTAN (ours) |
|---|---|---|
| Edge criterion | Feature similarity (Euclidean) | Shared attribute (fraud ring) |
| Direction | Undirected | Temporal (past→future) |
| Scale | 20% sample | All 590K transactions |
| Fraud ring capture | No | Yes — same card/device |
| Literature basis | None | GTAN AAAI 2023 |

In [ ]:
def build_gtan_graph(n_nodes, relation_series_dict, k=3, max_group=5000):
    """
    Build GTAN-style homogeneous transaction graph.

    For each relation type:
      - Group transactions by shared attribute value
      - For each transaction at position i in the group, add directed edges
        from the up-to-k most recent preceding transactions in the same group
      - Edges are past → future (temporal ordering preserved)

    After collecting all edges:
      - Add reverse edges (dgl.add_reverse_edges) for bidirectional message passing
      - Add self-loops (dgl.add_self_loop)

    Args:
        n_nodes: total number of transaction nodes
        relation_series_dict: dict mapping relation_name → pd.Series of group keys
                              (index = global transaction index, values = group key)
        k: max temporal neighbors per transaction per relation
        max_group: groups larger than this are skipped (avoids hub nodes)

    Returns:
        DGLGraph with n_nodes nodes and temporal relation edges
    """
    src_list, dst_list = [], []
    stats = {}

    for rel_name, series in relation_series_dict.items():
        n_edges_before = len(src_list)
        skipped_large = 0
        skipped_nan   = 0

        # Group by attribute value; data is already sorted by TransactionDT
        for group_val, group_idx in series.groupby(series).groups.items():
            # Skip NaN / sentinel groups
            if pd.isna(group_val) or group_val in (-1, '-1', 'nan'):
                skipped_nan += 1
                continue
            indices = list(group_idx)
            n = len(indices)
            if n < 2:
                continue
            # Skip overly broad groups (e.g. gmail.com)
            if n > max_group:
                skipped_large += 1
                continue
            # Connect each transaction to its k nearest past transactions in group
            for i in range(1, n):
                dst = indices[i]
                start = max(0, i - k)
                for src in indices[start:i]:
                    src_list.append(src)
                    dst_list.append(dst)

        n_added = len(src_list) - n_edges_before
        stats[rel_name] = {'edges': n_added, 'skipped_large': skipped_large, 'skipped_nan': skipped_nan}
        print(f'  [{rel_name}] edges={n_added:,}  skipped_large={skipped_large}  skipped_nan={skipped_nan}')

    if len(src_list) == 0:
        print('  WARNING: No edges built. Check relation columns.')
        g = dgl.graph(([], []), num_nodes=n_nodes)
    else:
        src_arr = np.array(src_list, dtype=np.int64)
        dst_arr = np.array(dst_list, dtype=np.int64)
        g = dgl.graph((src_arr, dst_arr), num_nodes=n_nodes)

    g = dgl.add_reverse_edges(g)
    g = dgl.add_self_loop(g)

    print(f'\nFinal graph: {g.num_nodes():,} nodes, {g.num_edges():,} edges')
    print(f'Avg degree: {g.num_edges() / g.num_nodes():.1f}')
    return g, stats

print('Graph builder defined.')

In [ ]:
# ── Build GTAN graph on all training transactions ──────────────────────────────
# Data is already sorted by TransactionDT (done in preprocessing)
# Relation series: index = global row index (0 ... 590539)
relation_series = {
    'uid':          uid_train.reset_index(drop=True),
    'card1':        card1_train.reset_index(drop=True),
    'P_emaildomain': pemail_train.reset_index(drop=True),
    'DeviceInfo':   device_train.reset_index(drop=True),
}

print(f'Building GTAN graph on {len(y_gat):,} transactions...')
print(f'K={GRAPH_K_NEIGHBORS} temporal neighbors, max_group={MAX_GROUP_SIZE}')
print()

t0 = time.time()
g, graph_stats = build_gtan_graph(
    n_nodes=len(y_gat),
    relation_series_dict=relation_series,
    k=GRAPH_K_NEIGHBORS,
    max_group=MAX_GROUP_SIZE
)
print(f'\nGraph built in {time.time()-t0:.1f}s')

# Attach features and labels to graph
g.ndata['feat']  = node_features
g.ndata['label'] = node_labels

# Save graph stats
with open(os.path.join(SAVE_DIR, 'graph_stats.json'), 'w') as f:
    json.dump({
        'n_nodes': g.num_nodes(),
        'n_edges': g.num_edges(),
        'avg_degree': g.num_edges() / g.num_nodes(),
        'relation_stats': graph_stats
    }, f, indent=2)

In [ ]:
# ── Graph statistics & fraud connectivity analysis ─────────────────────────────
in_deg = g.in_degrees().numpy()
out_deg = g.out_degrees().numpy()
total_deg = in_deg + out_deg

fraud_mask = y_gat == 1
benign_mask = y_gat == 0

print('=== Graph Statistics ===')
print(f'Nodes:            {g.num_nodes():>10,}')
print(f'Edges:            {g.num_edges():>10,}')
print(f'Avg total degree: {total_deg.mean():>10.2f}')
print(f'Max total degree: {total_deg.max():>10,}')
print()
print(f'Fraud node avg degree:   {total_deg[fraud_mask].mean():.2f}')
print(f'Benign node avg degree:  {total_deg[benign_mask].mean():.2f}')
print(f'Fraud nodes with deg>2:  {(total_deg[fraud_mask] > 2).mean()*100:.1f}%')

# Degree distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.clip(total_deg, 0, 50), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Degree Distribution (clipped at 50)')
axes[0].set_xlabel('Degree')
axes[0].set_ylabel('Count')

axes[1].hist(np.clip(total_deg[fraud_mask], 0, 50), bins=30, alpha=0.7, label='Fraud', color='red')
axes[1].hist(np.clip(total_deg[benign_mask], 0, 50), bins=30, alpha=0.5, label='Benign', color='blue')
axes[1].set_title('Degree Distribution by Label')
axes[1].set_xlabel('Degree')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'degree_distribution.png'), dpi=100)
plt.show()

---
## 3. GAT Model Architecture

**Based on:** Veličković et al., "Graph Attention Networks" (ICLR 2018) + Architecture Justification v2 §4.4

**Architecture:**
```
Input: (N, in_dim=263)
  Layer 1: GATConv(263 → 64, heads=4, ELU, BatchNorm)
  flatten → (N, 256)
  Dropout(0.3)
  Layer 2: GATConv(256 → 64, heads=1, no activation, BatchNorm, residual)
  squeeze → (N, 64)  ← relational embedding for fusion
  ↓
  FraudClassifier: Linear(64→32) → ReLU → Dropout(0.2) → Linear(32→1)
```

**Design choices (literature-validated):**
- 2 layers: matches GTAN/CaT-GNN; avoids over-smoothing (Li et al., AAAI 2018)
- 4 heads: CaT-GNN uses 4; provides diversity for attention-based causal scoring in CaT-GNN downstream
- `hidden_dim=64`: consistent with TCN (64) and CaT-GNN (64) for fusion compatibility
- ELU activation: recommended by Veličković et al. for GAT
- `allow_zero_in_degree=True`: isolated transactions exist in sparse graphs
- Mini-batch with `fan_out=[10, 5]`: samples 10 neighbors at L1, 5 at L2 → ~50 nodes per target

In [ ]:
class FraudGAT(nn.Module):
    """
    2-layer Graph Attention Network for fraud detection.
    Architecture from architecture_justification_v2 §4.4:
      Layer 1: in_dim → hidden_dim × num_heads  (multi-head, ELU, BatchNorm)
      Layer 2: hidden_dim×num_heads → hidden_dim (single head, residual, BatchNorm)
    Output: (N, hidden_dim=64) relational embedding — compatible with Gated Fusion.

    Why 2 layers for 263 features:
      - Layers control hop depth, NOT feature capacity.
        263 features → 64 per head × 4 heads = 256-d projection is already expressive.
      - 3 layers on a graph with avg degree ~8 reaches 8³≈512 nodes/target → over-smoothing.
      - GTAN (1 layer), CaT-GNN (1 layer internal GAT), CARE-GNN (1-2 layers).

    Why 4 heads (not 8 as in v1 doc):
      - v1 proposed 8 heads for 74 features on a heterogeneous graph (different project).
      - 8 heads × 64 = 512-d intermediate > input dim (263) → over-parametrized.
      - CaT-GNN downstream REQUIRES 4 heads: averages attention across 4 heads for
        causal/environment split (Eq 2). Changing to 8 would break CaT-GNN compatibility.
    """
    def __init__(self, in_dim, hidden_dim=64, num_heads=4,
                 feat_drop=0.3, attn_drop=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads

        # Layer 1: in_dim → hidden_dim × num_heads
        self.gat1 = GATConv(
            in_feats=in_dim,
            out_feats=hidden_dim,
            num_heads=num_heads,
            feat_drop=feat_drop,
            attn_drop=attn_drop,
            activation=F.elu,
            residual=False,
            allow_zero_in_degree=True
        )
        self.bn1 = nn.BatchNorm1d(hidden_dim * num_heads)

        # Layer 2: hidden_dim×num_heads → hidden_dim (single head, residual)
        self.gat2 = GATConv(
            in_feats=hidden_dim * num_heads,
            out_feats=hidden_dim,
            num_heads=1,
            feat_drop=feat_drop,
            attn_drop=attn_drop,
            activation=None,
            residual=True,   # DGL auto-adds res_fc: (256 → 64) projection
            allow_zero_in_degree=True
        )
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.drop = nn.Dropout(feat_drop)

    def forward(self, blocks, x):
        """Mini-batch forward — accepts DGLBlocks from NeighborSampler."""
        h = self.gat1(blocks[0], x)   # (N_dst_0, num_heads, hidden_dim)
        h = h.flatten(1)               # (N_dst_0, num_heads * hidden_dim)
        h = self.bn1(h)
        h = self.drop(h)
        h = self.gat2(blocks[1], h)   # (N_dst_1, 1, hidden_dim)
        h = h.squeeze(1)               # (N_dst_1, hidden_dim)
        h = self.bn2(h)
        return h

    def forward_full(self, g, x):
        """Full-graph forward — for validation with MultiLayerFullNeighborSampler."""
        h = self.gat1(g, x)
        h = h.flatten(1)
        h = self.bn1(h)
        h = self.drop(h)
        h = self.gat2(g, h)
        h = h.squeeze(1)
        h = self.bn2(h)
        return h


class FraudClassifier(nn.Module):
    """MLP head: 64 → 32 → 1 (logit). Same structure as arch_justification_v2 §4.7."""
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, h):
        return self.fc(h)


# ── Loss function — from experiment_tcn (binary_crossentropy + class_weight='balanced') ──
# experiment_tcn uses compute_class_weight('balanced') → equivalent to
# pos_weight = n_benign / n_fraud in PyTorch's BCEWithLogitsLoss
from sklearn.utils.class_weight import compute_class_weight

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_gat
)
# class_weights_arr[1] = weight for fraud class
# In PyTorch BCE: pos_weight = (weight for class 1) / (weight for class 0)
pos_weight_val = torch.tensor([class_weights_arr[1] / class_weights_arr[0]], dtype=torch.float32)

def weighted_bce_loss(logits, targets):
    """
    Binary cross-entropy with pos_weight — mirrors experiment_tcn's
    binary_crossentropy + class_weight='balanced'.
    pos_weight = n_benign / n_fraud ≈ 27.4 for IEEE-CIS (3.5% fraud rate).
    """
    pw = pos_weight_val.to(logits.device)
    return F.binary_cross_entropy_with_logits(logits, targets.float(), pos_weight=pw)

# ── Instantiate models ────────────────────────────────────────────────────────
gat_model  = FraudGAT(
    in_dim=IN_DIM, hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS_L1, feat_drop=FEAT_DROP, attn_drop=ATTN_DROP
).to(DEVICE)
classifier = FraudClassifier(hidden_dim=HIDDEN_DIM).to(DEVICE)

n_params_gat  = sum(p.numel() for p in gat_model.parameters())
n_params_cls  = sum(p.numel() for p in classifier.parameters())
print(f'GAT parameters:        {n_params_gat:,}')
print(f'Classifier parameters: {n_params_cls:,}')
print(f'Total parameters:      {n_params_gat + n_params_cls:,}')
print(f'pos_weight (fraud):    {pos_weight_val.item():.2f}  '
      f'(n_benign/n_fraud = {class_weights_arr[1]/class_weights_arr[0]:.2f})')
print(f'Loss: binary_crossentropy + balanced class weights  [from experiment_tcn]')

---
## 4. Training Pipeline

Mini-batch training with DGL `NeighborSampler([10, 5])`:
- At each iteration, for each target node DGL samples 10 neighbors at Layer 1 and 5 at Layer 2
- Creates bipartite `DGLBlock` objects where `blocks[0]` handles L1, `blocks[1]` handles L2
- Validation uses `MultiLayerFullNeighborSampler` (all neighbors, batched) to get exact metrics

In [ ]:
# ── Optimizer & Scheduler — from experiment_tcn ───────────────────────────────
# experiment_tcn: Adam(lr=0.0005), ReduceLROnPlateau(patience=10, factor=0.5, min_lr=1e-7)
optimizer = torch.optim.Adam(
    list(gat_model.parameters()) + list(classifier.parameters()),
    lr=LR,           # 0.0005 — from experiment_tcn
    weight_decay=1e-5
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',          # monitor val_auc (higher = better)
    factor=LR_FACTOR,    # 0.5 — from experiment_tcn
    patience=PATIENCE_LR,# 10  — from experiment_tcn
    min_lr=MIN_LR,       # 1e-7 — from experiment_tcn
    verbose=True
)

# ── Dataloaders ───────────────────────────────────────────────────────────────
train_sampler = dgl.dataloading.NeighborSampler(FAN_OUT)
train_loader  = dgl.dataloading.DataLoader(
    g, train_nids, train_sampler,
    batch_size=BATCH_SIZE,  # 512 — from experiment_tcn
    shuffle=True,
    drop_last=False,
    num_workers=0,
    device=DEVICE
)

# Validation: MultiLayerFullNeighborSampler (all neighbors, no sampling noise)
val_sampler = dgl.dataloading.MultiLayerFullNeighborSampler(2)
val_loader  = dgl.dataloading.DataLoader(
    g, val_nids, val_sampler,
    batch_size=2048,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    device=DEVICE
)

print(f'Optimizer: Adam  lr={LR}  weight_decay=1e-5')
print(f'Scheduler: ReduceLROnPlateau  patience={PATIENCE_LR}  factor={LR_FACTOR}  min_lr={MIN_LR}')
print(f'Train batches: {len(train_loader):,}  |  Val batches: {len(val_loader):,}')

In [ ]:
def train_epoch(loader):
    gat_model.train()
    classifier.train()
    total_loss, n_batches = 0.0, 0

    for input_nodes, output_nodes, blocks in loader:
        x = blocks[0].srcdata['feat']
        y = node_labels[output_nodes.cpu()].to(DEVICE)

        h      = gat_model(blocks, x)
        logits = classifier(h).squeeze(-1)
        # Weighted BCE — mirrors experiment_tcn binary_crossentropy + class_weight='balanced'
        loss   = weighted_bce_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(gat_model.parameters()) + list(classifier.parameters()), 1.0
        )
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(loader):
    gat_model.eval()
    classifier.eval()
    all_probs, all_labels = [], []

    for input_nodes, output_nodes, blocks in loader:
        x      = blocks[0].srcdata['feat']
        h      = gat_model(blocks, x)
        logits = classifier(h).squeeze(-1)
        probs  = torch.sigmoid(logits).cpu().numpy()
        labels = node_labels[output_nodes.cpu()].numpy()
        all_probs.extend(probs)
        all_labels.extend(labels)

    probs  = np.array(all_probs)
    labels = np.array(all_labels)

    auc       = roc_auc_score(labels, probs)
    y_pred    = (probs >= 0.5).astype(int)
    f1        = f1_score(labels, y_pred, zero_division=0)
    precision = precision_score(labels, y_pred, zero_division=0)
    recall    = recall_score(labels, y_pred, zero_division=0)

    return {
        'auc': float(auc), 'f1': float(f1),
        'precision': float(precision), 'recall': float(recall)
    }, probs, labels

print('Train / evaluate functions defined.')
print('Loss: weighted_bce_loss (binary_crossentropy + balanced weights)  [from experiment_tcn]')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
# experiment_tcn: EarlyStopping(monitor='val_auc', mode='max', patience=50)
#                 ModelCheckpoint(monitor='val_auc', save_best_only=True)
history = {'train_loss': [], 'val_auc': [], 'val_f1': [], 'val_precision': [], 'val_recall': []}
best_auc   = 0.0
no_improve = 0
best_probs = best_labels = None

print(f'Training for up to {EPOCHS} epochs  |  EarlyStopping patience={PATIENCE_ES}  |  monitor=val_auc')
print(f'{"Epoch":>6} {"TrainLoss":>10} {"ValAUC":>8} {"ValF1":>8} {"ValPrec":>8} {"ValRecall":>9}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss = train_epoch(train_loader)
    val_metrics, val_probs, val_labels_ep = evaluate(val_loader)

    scheduler.step(val_metrics['auc'])

    history['train_loss'].append(train_loss)
    history['val_auc'].append(val_metrics['auc'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])

    # Save best model (mirrors ModelCheckpoint save_best_only=True)
    if val_metrics['auc'] > best_auc:
        best_auc    = val_metrics['auc']
        no_improve  = 0
        best_probs  = val_probs.copy()
        best_labels = val_labels_ep.copy()
        torch.save(gat_model.state_dict(),  os.path.join(SAVE_DIR, 'models', 'gat_best.pt'))
        torch.save(classifier.state_dict(), os.path.join(SAVE_DIR, 'models', 'classifier_best.pt'))
        marker = ' *'
    else:
        no_improve += 1
        marker = ''

    if epoch % 10 == 0 or epoch == 1:
        elapsed = time.time() - t0
        print(f'{epoch:>6} {train_loss:>10.4f} '
              f'{val_metrics["auc"]:>8.4f} {val_metrics["f1"]:>8.4f} '
              f'{val_metrics["precision"]:>8.4f} {val_metrics["recall"]:>9.4f}  '
              f'({elapsed:.1f}s){marker}')

    # EarlyStopping — patience=50, monitor=val_auc (same as experiment_tcn)
    if no_improve >= PATIENCE_ES:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE_ES} epochs)')
        break

print(f'\nBest Val AUC: {best_auc:.4f}  (epoch {int(np.argmax(history["val_auc"]))+1})')
with open(os.path.join(SAVE_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

---
## 5. Evaluation & Results

In [ ]:
# ── Optimal threshold search ──────────────────────────────────────────────────
# Default 0.5 is suboptimal at 3.5% fraud rate; search for best F1 threshold
gat_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'models', 'gat_best.pt')))
classifier.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'models', 'classifier_best.pt')))

_, best_probs, best_labels = evaluate(val_loader)

thresholds = np.arange(0.1, 0.95, 0.01)
f1_scores  = [f1_score(best_labels, (best_probs >= t).astype(int), zero_division=0)
              for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]
best_f1     = max(f1_scores)

print(f'Optimal threshold: {best_thresh:.2f}  (F1={best_f1:.4f})')

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores)
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best: {best_thresh:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs. Classification Threshold')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'threshold_search.png'), dpi=100)
plt.show()

In [ ]:
# ── Final metrics at optimal threshold ───────────────────────────────────────
y_pred_opt = (best_probs >= best_thresh).astype(int)

auc       = roc_auc_score(best_labels, best_probs)
f1        = f1_score(best_labels, y_pred_opt, zero_division=0)
precision = precision_score(best_labels, y_pred_opt, zero_division=0)
recall    = recall_score(best_labels, y_pred_opt, zero_division=0)
cm        = confusion_matrix(best_labels, y_pred_opt)

print('=== Final Validation Metrics (Best Model) ===')
print(f'  AUC-ROC:          {auc:.4f}')
print(f'  F1 Score:         {f1:.4f}  (threshold={best_thresh:.2f})')
print(f'  Precision:        {precision:.4f}')
print(f'  Recall:           {recall:.4f}')
print()
print('=== Confusion Matrix ===')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')
print()
print(classification_report(best_labels, y_pred_opt, target_names=['Benign', 'Fraud']))

# Save results
results = {
    'best_epoch': int(np.argmax(history['val_auc'])) + 1,
    'best_val_auc': float(auc),
    'best_val_f1': float(f1),
    'best_val_precision': float(precision),
    'best_val_recall': float(recall),
    'optimal_threshold': float(best_thresh),
    'confusion_matrix': cm.tolist(),
    'n_train': len(train_nids),
    'n_val': len(val_nids),
    'in_dim': IN_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_heads': NUM_HEADS_L1,
    'graph_nodes': g.num_nodes(),
    'graph_edges': g.num_edges(),
}
with open(os.path.join(SAVE_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved.')

In [ ]:
# ── Training curves & confusion matrix ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0, 0].plot(epochs_range, history['train_loss'], label='Train Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Focal Loss')
axes[0, 0].legend()

axes[0, 1].plot(epochs_range, history['val_auc'], label='Val AUC', color='green')
axes[0, 1].axhline(best_auc, color='red', linestyle='--', label=f'Best: {best_auc:.4f}')
axes[0, 1].set_title('Validation AUC-ROC')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('AUC')
axes[0, 1].legend()

axes[1, 0].plot(epochs_range, history['val_f1'],        label='F1')
axes[1, 0].plot(epochs_range, history['val_precision'], label='Precision')
axes[1, 0].plot(epochs_range, history['val_recall'],    label='Recall')
axes[1, 0].set_title('Val F1 / Precision / Recall (threshold=0.5)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Benign', 'Pred Fraud'],
    yticklabels=['True Benign', 'True Fraud'],
    ax=axes[1, 1]
)
axes[1, 1].set_title(f'Confusion Matrix (threshold={best_thresh:.2f})')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_results.png'), dpi=100)
plt.show()

In [ ]:
# ── Compare GAT vs TCN (experiment_tcn results for reference) ────────────────
TCN_RESULTS = {
    'AUC':       0.9659,
    'F1':        0.4474,
    'Precision': 0.2993,
    'Recall':    0.8851,
}
GAT_RESULTS = {
    'AUC':       auc,
    'F1':        f1,
    'Precision': precision,
    'Recall':    recall,
}

metrics = ['AUC', 'F1', 'Precision', 'Recall']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, [TCN_RESULTS[m] for m in metrics], width,
               label='TCN (experiment_tcn)', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, [GAT_RESULTS[m] for m in metrics], width,
               label='GAT (GTAN graph, this notebook)', color='darkorange', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('TCN vs GAT — Validation Performance\n(Different views: temporal vs relational)')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'tcn_vs_gat.png'), dpi=100)
plt.show()

print('Note: GAT and TCN capture complementary signals.')
print('TCN = temporal velocity/burst patterns.')
print('GAT = relational fraud-ring structure.')
print('Both will be fused with CaT-GNN via Gated Attention Fusion in the final pipeline.')

---
## 6. Summary

### What was built
- **GTAN-style homogeneous transaction graph**: 590K nodes, edges via `uid/card1/P_emaildomain/DeviceInfo`, K=3 temporal neighbors, forward-in-time direction
- **2-layer GAT**: `263 → 64×4h=256 → 64×1h=64`, Focal Loss, mini-batch NeighborSampler([10, 5])
- **Output**: 64-d relational embedding per transaction, compatible with the Gated Fusion layer

### Key differences from TCN-pipeline GAT (and why they matter)
| Aspect | TCN Pipeline | This Notebook |
|--------|-------------|---------------|
| Graph edges | kNN Euclidean (20% sample) | GTAN temporal/relational (100%, all 590K) |
| Message passing | None (single-node per sample) | Full 2-hop neighborhood aggregation |
| Node features | Flattened sequences (failed approach) | Flat 263-feature vector |
| Output dim | 32 (mismatches fusion) | 64 (matches TCN + CaT-GNN) |
| Framework | TF/Keras | PyTorch + DGL |

### Next steps toward full pipeline
1. **CaT-GNN notebook**: Use the same GTAN graph + same 263 features; CaT-GNN Inspector runs GATConv on top, splits neighbors into causal/environment sets, Intervener applies learned blending
2. **Gated Fusion**: Stack TCN (64-d) + GAT (64-d) + CaT-GNN (64-d) → 192-d concat → gating weights → 64-d fused embedding
3. **XGBoost Meta-Learner**: Stack 64-d fused embedding + original 263 tabular features

### Validated architecture choices
- 2 layers / 4 heads matches GTAN/CaT-GNN literature; avoids over-smoothing
- `hidden_dim=64` gives 64-d output consistent across all three views
- GTAN graph construction is literature-backed (AAAI 2023) and fraud-ring aware
- DGL is the correct framework: direct GTAN code compatibility, fraud research standard